# Guía del flujo modelo: consulta de reparación

Esta guía explica el código existente para que puedas construir los demás flujos. El plan está en `PLAN_CHATBOT.ipynb`; el contrato HTTP propuesto, en `CONTRATO_API.ipynb`. Las referencias a HU/RF y fechas de ese plan están pendientes de contrastar nuevamente: los tres PDF originales no están disponibles en la nueva ruta del proyecto.



## Recorrido de la conversación

```mermaid
flowchart TD
    A[Cliente escribe estado] --> B[Bot solicita folio]
    B --> C{Respuesta}
    C -->|cancelar| D[Termina sin consultar]
    C -->|Formato inválido| E[Explica formato y vuelve a capturar]
    E --> C
    C -->|Folio válido| F[Consulta servicio con folio y remitente Meta]
    F --> G{Resultado}
    G -->|encontrado| H[Muestra campos públicos y termina]
    G -->|no_autorizado| I[Respuesta genérica y termina]
    G -->|no_disponible| J[Informa indisponibilidad y termina]
```



## Cómo leer cada archivo

1. **`src/config.ts`**: `requerida` comprueba variables sin revelar valores. `cargarConfiguracion` valida puerto y modo, prepara opciones de Meta y exige credenciales de Backend sólo en modo API. No configura la aplicación de Meta: eso se hace en su panel.
2. **`src/app.ts`**: selecciona el servicio demo o HTTP, crea `MetaProvider`, registra el flujo con `createFlow` y arranca el servidor. Es el lugar donde conectarás un flujo nuevo. `MemoryDB` conserva historial temporal; la información real del servicio pertenece al Backend.
3. **`src/flows/consulta.flow.ts`**: `crearFlujoConsulta` recibe la función `consultar` y la bandera demo. `addKeyword` define entradas; `addAnswer` muestra la pregunta y `capture: true` espera la respuesta. El callback usa `ctx.body` para el folio y `ctx.from` para el remitente. Primero permite cancelar, después valida y finalmente consulta. `fallBack` vuelve a pedir el dato; `endFlow` envía la respuesta final. Las opciones están descritas en la [documentación de funciones de BuilderBot](https://www.builderbot.app/en/add-functions).
4. **`src/services/consultar-servicio.ts`**: define estados y datos públicos, normaliza el folio y valida respuestas externas. `crearConsultaApi` envía folio y teléfono con token del bot y timeout de cinco segundos. `leerServicio` selecciona los campos que se pueden mostrar. `crearConsultaDemo` devuelve un caso ficticio únicamente para el remitente configurado.
5. **`tests/consulta.test.ts`**: comprueba folios, selección de campos, restricción demo y manejo del transporte HTTP. Los mocks no verifican autorización real del Backend ni entrega por WhatsApp.



## Ejemplo para probar

Con `CHATBOT_MODE=demo`, variables de Meta completas y `DEMO_PHONE_NUMBER` configurado:

```text
Cliente: estado
Bot: Bienvenido a Fixify... Escribe tu folio o cancelar.
Cliente: DEMO-001
Bot: DATOS FICTICIOS — DEMOSTRACIÓN
     Folio: DEMO-001
     Equipo: Laptop de demostración
     Estado: En revisión
     Última actualización: 2026-10-04T12:00:00-06:00
```

Comprueba también `AB` (repite captura), `cancelar` (termina) y un folio diferente (respuesta genérica). En modo API comprueba Backend caído, respuesta inválida y servicio ajeno. El endpoint propuesto todavía requiere implementación e integración; no se considera finalizada la consulta real.



## Patrón para tu siguiente flujo

Antes de escribirlo, completa esta ficha:

| Parte | Qué documentar |
| --- | --- |
| Requisito | HU/RF y página del documento que lo sustenta |
| Entrada | Palabras que activan el flujo y datos solicitados |
| Validación | Formato, cancelación, reintentos y caducidad |
| Servicio | Contrato, identidad, permisos y respuesta pública |
| Salida | Mensajes de éxito, dato desconocido y fallo técnico |
| Evidencia | Conversación y pruebas; limitaciones pendientes |

Luego crea un servicio tipado y una fábrica de flujo en archivos separados. Inyecta el servicio desde `app.ts` y registra el flujo junto a `crearFlujoConsulta(...)` en `createFlow`. Usa palabras de entrada distintas. Si capturas varios datos, guarda sólo los necesarios en `state`, valida cada paso y elimina los datos al cancelar o concluir. Sigue el mismo orden: capturar → validar → consultar → responder → cerrar.

Documenta cada función con su propósito, entradas, resultado, errores y requisito asociado. En los comentarios interiores explica decisiones, por ejemplo por qué folio inexistente y servicio ajeno reciben el mismo mensaje. Evita duplicar en comentarios lo que ya expresa una línea de código.



## Comprobación del entorno

Ejecuta `npm test`, `npm run lint` y `npm run build` desde `chatbot/base-ts-meta-postgres`. Si los lanzadores buscan `node.exe` en Linux, las dependencias fueron copiadas desde otro sistema; reinstálalas en Linux con `npm ci`. No cambies el código del flujo para corregir ese problema de instalación.

La configuración del proveedor y webhook se describe en la [guía oficial de Meta en BuilderBot](https://www.builderbot.app/en/providers/meta). No uses sus ejemplos antiguos de versión o tarifas como configuración vigente de tu cuenta.
